### Read Bronze Table

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.window import Window

df_bronze = spark.table(
    "fintech.bronze.stock_prices"
)

display(df_bronze)

### Quality rules

In [0]:
df_validated = (
    df_bronze
    .filter(col("symbol").isNotNull())
    .filter(col("trade_date").isNotNull())
    .filter(col("open") > 0)
    .filter(col("high") > 0)
    .filter(col("low") > 0)
    .filter(col("close") > 0)
    .filter(col("volume") > 0)
    .filter(col("high") >= col("open"))
    .filter(col("high") >= col("close"))
    .filter(col("low") <= col("open"))
    .filter(col("low") <= col("close"))
)

### Deduplication

symbol + trade_date

In [0]:
window = (
    Window
    .partitionBy("symbol", "trade_date")
    .orderBy(col("_ingestion_timestamp").desc())
)

df_silver = (
    df_validated
    .withColumn(
        "_row_number",
        row_number().over(window)
    )
    .filter(
        col("_row_number") == 1
    )
    .drop("_row_number")
)

In [0]:
print("Validated:", df_validated.count())
print("Silver:", df_silver.count())

In [0]:
duplicates = (
    df_silver
    .groupBy("symbol", "trade_date")
    .count()
    .filter(col("count")>1)
)

display(duplicates)

### Merge

In [0]:
print(
    spark.table(
        "fintech.silver.stock_prices"
    ).count()
)

In [0]:
from delta.tables import DeltaTable 

# Ref to the UC table
silver_table = DeltaTable.forName(
    spark,
    "fintech.silver.stock_prices"
)

# Merge exe
(
    silver_table.alias("target")
    .merge(
        df_silver.alias("source"),
        """
        target.symbol = source.symbol
        AND target.trade_date = source.trade_date
        """
    )
    .whenMatchedUpdate(
        condition="""
            source._ingestion_timestamp > target._ingestion_timestamp
        """,
        set={
            "open": "source.open",
            "high": "source.high",
            "low": "source.low",
            "close": "source.close",
            "volume": "source.volume",
            "change": "source.change",
            "change_percent": "source.change_percent",
            "vwap": "source.vwap",
            "_ingestion_timestamp": "source._ingestion_timestamp",
            "_source_file": "source._source_file"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
df_silver_result = spark.table(
        "fintech.silver.stock_prices"
    )

print(f"Silver records: {df_silver_result.count()}")

In [0]:
display(
    df_silver_result
    .groupBy("symbol")
    .count()
    .orderBy("symbol")
)

In [0]:
duplicates = (
    df_silver_result
    .groupBy("symbol", "trade_date")
    .count()
    .filter("count > 1")
)

print(f"Duplicate business keys: {duplicates.count()}")

In [0]:
"""
# Bootstrap Inicial
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fintech.silver.stock_prices")
)
"""